# Using Kerchunk and Icechunk with grib2io and Xarray

This notebook demonstrates how to virtualize GRIB2 files using Kerchunk and store them in a cloud-native, versioned format using Icechunk.

In [ ]:
import grib2io
import xarray as xr
import pathlib
import os
from grib2io.kerchunk import ReferenceGenerator
from grib2io.icechunk import open_grib2, open_dataset

## 1. Generate Kerchunk References
We use `ReferenceGenerator` to scan a GRIB2 file and create a manifest of byte-range references.

In [2]:
# Use the GFS test file bundled with grib2io.
# Assumes the notebook is run from the project root or demos/ directory.
_here = pathlib.Path(os.path.abspath(""))
_candidates = [
    _here / "tests" / "input_data" / "gfs.t00z.pgrb2.1p00.f024",
    _here.parent / "tests" / "input_data" / "gfs.t00z.pgrb2.1p00.f024",
]
grib2_file = next(str(p) for p in _candidates if p.exists())

gen = ReferenceGenerator(grib2_file)
manifest = gen.generate()
print(f"Generated manifest with {len(manifest['refs'])} references")

Generated manifest with 1225 references


## 2. Open as an xarray Dataset

`open_dataset` writes the manifest into a temporary Icechunk virtual store and opens it
with `xarray.open_zarr` in one call. It handles both local files and remote object stores —
virtual chunk authorization is set up automatically from the URI prefixes in the manifest.

For an even shorter path when you have the file URL directly, use `open_grib2(url)`.


In [ ]:
ds = open_dataset(manifest)
display(ds)

  2026-05-21T16:24:00.997547Z  WARN icechunk_arrow_object_store: The LocalFileSystem storage is not safe for concurrent commits. If more than one thread/process will attempt to commit at the same time, prefer using object stores.
    at icechunk-arrow-object-store/src/lib.rs:196

Committed snapshot: Y93810SEH59E41DYZBZG


## 4. Efficient Variable Streaming
Because the data is lazily indexed, we can "stream" or access specific parts of a variable without loading the entire dataset into memory. Using `chunks={}` in `open_dataset` ensures that Xarray uses Dask for lazy loading.

This is particularly powerful for large variables like temperature (TMP) or humidity (RH) across multiple vertical levels or time steps.

In [5]:
if "TMP" in ds.data_vars:
    # Select a spatial subset — only triggers byte-range requests for the relevant GRIB messages.
    tmp_subset = ds.TMP.isel(y=slice(0, 100), x=slice(0, 100))

    # Trigger a compute to see the data
    data = tmp_subset.compute()
    print("Successfully streamed TMP subset data.")
    display(data)

IcechunkError:   x a virtual chunk in this repository resolves to the url prefix file:///Users/barry/Documents/grib2io/tests/input_data/, to be able to fetch the chunk you need to authorize the virtual chunk
  | container when you open/create the repository, see https://icechunk.io/en/stable/virtual/
  | 
  | context:
  |    0: icechunk::store::get
  |            with key="TMP/c/0/7/0/0" byte_range=From(0)
  |              at icechunk/src/store.rs:183
  | 
  `-> a virtual chunk in this repository resolves to the url prefix file:///Users/barry/Documents/grib2io/tests/input_data/, to be able to fetch the chunk you need to authorize the virtual chunk
      container when you open/create the repository, see https://icechunk.io/en/stable/virtual/


## 5. Multiple Variable Streaming
We can also stream multiple variables simultaneously. Dask will manage the parallel fetch of these chunks from the underlying storage (e.g. S3 or local disk) via the Icechunk references.

In [ ]:
# Select multiple variables for a specific region
available = [v for v in ["TMP", "HGT"] if v in ds.data_vars]
if available:
    subset_ds = ds[available].isel(y=slice(50, 150), x=slice(50, 150))

    # Dask manages parallel byte-range fetches from the Icechunk store.
    computed_result = subset_ds.compute()
    print(f"Streamed {available} successfully.")
    display(computed_result)